In [1]:
import cv2
import mediapipe as mp
import math

from mediapipe.tasks.python import vision
from mediapipe.tasks import python

# ---------- Load Model ----------
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1
)

detector = vision.HandLandmarker.create_from_options(options)

# ---------- Helper ----------
def get_distance(p1, p2):
    return math.sqrt((p1.x - p2.x)**2 + (p1.y - p2.y)**2)

# ---------- Gesture Detection ----------
def detect_gesture(hand_landmarks):
    thumb_tip = hand_landmarks[4]
    index_tip = hand_landmarks[8]
    index_pip = hand_landmarks[6]

    middle_tip = hand_landmarks[12]
    middle_pip = hand_landmarks[10]

    ring_tip = hand_landmarks[16]
    pinky_tip = hand_landmarks[20]

    pinch_dist = get_distance(thumb_tip, index_tip)

    # Fold detection
    index_folded = index_tip.y > index_pip.y
    middle_folded = middle_tip.y > middle_pip.y
    ring_folded = ring_tip.y > middle_pip.y
    pinky_folded = pinky_tip.y > middle_pip.y

    folded_count = sum([index_folded, middle_folded, ring_folded, pinky_folded])

    if pinch_dist < 0.05 and folded_count < 4:
        return "PINCH"
    elif folded_count >= 3:
        return "FIST"
    else:
        return "OPEN_HAND"

# ---------- Gesture Stabilization ----------
gesture_history = []
history_length = 5
previous_gesture = ""

# ---------- Movement Tracking ----------
prev_x = None
movement_threshold = 0.02

# ---------- Camera ----------
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    result = detector.detect(mp_image)

    gesture = "NO_HAND"
    movement = "STILL"

    if result.hand_landmarks:
        hand_landmarks = result.hand_landmarks[0]

        # ----- Gesture -----
        gesture = detect_gesture(hand_landmarks)

        # ----- Movement -----
        wrist = hand_landmarks[0]
        current_x = wrist.x

        if prev_x is not None:
            dx = current_x - prev_x

            if dx > movement_threshold:
                movement = "RIGHT"
            elif dx < -movement_threshold:
                movement = "LEFT"
            else:
                movement = "STILL"

        prev_x = current_x

        # Draw landmarks
        h, w, _ = frame.shape
        for lm in hand_landmarks:
            cx, cy = int(lm.x * w), int(lm.y * h)
            cv2.circle(frame, (cx, cy), 4, (0, 255, 0), -1)

    # ---------- Stabilize Gesture ----------
    gesture_history.append(gesture)

    if len(gesture_history) > history_length:
        gesture_history.pop(0)

    stable_gesture = max(set(gesture_history), key=gesture_history.count)

    # ---------- Send Only on Change ----------
    if stable_gesture != previous_gesture:
        print("Gesture:", stable_gesture)

        # For Arduino later:
        # arduino.write(stable_gesture.encode())

        previous_gesture = stable_gesture

    # ---------- Display ----------
    cv2.putText(frame, f"Gesture: {stable_gesture}", (50, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)

    cv2.putText(frame, f"Move: {movement}", (50, 100),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)

    cv2.imshow("Gesture + Movement Control", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

Gesture: NO_HAND
Gesture: FIST
Gesture: OPEN_HAND
Gesture: NO_HAND
Gesture: OPEN_HAND
Gesture: FIST
Gesture: OPEN_HAND
Gesture: FIST
Gesture: OPEN_HAND
Gesture: NO_HAND
Gesture: OPEN_HAND
Gesture: NO_HAND
Gesture: FIST
Gesture: NO_HAND
Gesture: FIST
Gesture: OPEN_HAND
Gesture: NO_HAND
Gesture: OPEN_HAND
Gesture: PINCH
Gesture: FIST
Gesture: OPEN_HAND
Gesture: FIST
Gesture: PINCH
Gesture: FIST
Gesture: PINCH
Gesture: FIST
Gesture: NO_HAND
Gesture: FIST
Gesture: NO_HAND
Gesture: FIST
Gesture: NO_HAND
Gesture: FIST
Gesture: NO_HAND
Gesture: FIST
Gesture: NO_HAND
Gesture: FIST
Gesture: NO_HAND
Gesture: FIST
Gesture: NO_HAND
Gesture: FIST
Gesture: NO_HAND
Gesture: FIST
Gesture: NO_HAND
Gesture: FIST
Gesture: NO_HAND
Gesture: FIST
Gesture: NO_HAND
Gesture: FIST
Gesture: NO_HAND
Gesture: OPEN_HAND
Gesture: FIST
Gesture: NO_HAND
Gesture: OPEN_HAND
Gesture: FIST
Gesture: OPEN_HAND
Gesture: FIST
Gesture: OPEN_HAND
Gesture: FIST
Gesture: OPEN_HAND
Gesture: FIST
Gesture: OPEN_HAND
Gesture: FIST
Ge